# Comparison

## Loading Dataset

In [ ]:
from datasets import load_dataset

dataset = load_dataset("jhofff/handwriting-dataset")
all_data = dataset["train"]

train_samples = [{"image": x["image_path"], "text": x["text"]} for x in all_data if x["split"] == "train"]
val_samples   = [{"image": x["image_path"], "text": x["text"]} for x in all_data if x["split"] == "val"]
test_samples  = [{"image": x["image_path"], "text": x["text"]} for x in all_data if x["split"] == "test"]

print("Successfully loaded dataset")
print(f"Train: {len(train_samples)} | Val: {len(val_samples)} | Test: {len(test_samples)}")

## Loading Pretrained and Fine-tuned models

In [ ]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel, GenerationConfig
from PIL import Image, ImageOps
import torch
import os

# Load pretrained model
pretrained_processor = TrOCRProcessor.from_pretrained("microsoft/trocr-small-handwritten")
pretrained_model = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-small-handwritten", torch_dtype=torch.float32)
pretrained_model.eval()

# Load fine-tuned model from Hub
finetuned_processor = TrOCRProcessor.from_pretrained("jhofff/trocr-finetuned-handwriting")
finetuned_model = VisionEncoderDecoderModel.from_pretrained("jhofff/trocr-finetuned-handwriting", torch_dtype=torch.float32)
finetuned_model.eval()

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

pretrained_model.to(device)
finetuned_model.to(device)

# Fix token IDs on pretrained model
pretrained_model.config.decoder_start_token_id = 1
pretrained_model.config.pad_token_id = 1
pretrained_model.config.eos_token_id = 2
pretrained_model.generation_config = GenerationConfig(
    decoder_start_token_id=1,
    eos_token_id=2,
    pad_token_id=1,
    max_new_tokens=64,
    forced_eos_token_id=None,
    forced_bos_token_id=None,
)

def predict(model, processor, sample):
    image = sample["image"].convert("RGB")
    image = ImageOps.grayscale(image)
    image = ImageOps.autocontrast(image)
    image = image.convert("RGB")
    pixel_values = processor(images=image, return_tensors="pt").pixel_values.to(device).float()
    with torch.no_grad():
        generated_ids = model.generate(pixel_values)
    return processor.batch_decode(generated_ids, skip_special_tokens=True)[0]


def compare(image_path):
    pretrained_out = predict(pretrained_model, pretrained_processor, image_path)
    finetuned_out  = predict(finetuned_model,  finetuned_processor,  image_path)
    print(f"Image:      {os.path.basename(image_path)}")
    print(f"Pretrained: {pretrained_out}")
    print(f"Fine-tuned: {finetuned_out}")
    print()

print("Successfully loaded models")
print(f"device:", device)